### 02. Rotulagem da inativação e variáveis por domicílio

**Etapa CRISP-DM:** preparação dos dados.

**Definição de inativo (Fase 1):** domicílio que comprou na janela de observação e não comprou nas 8 semanas seguintes (sensibilidade com 4 e 12 semanas).

**Cortes (decisão de 16/11/2026):** observação de 26 semanas.
| Corte | Observação | Desempenho (rótulo) | Uso |
|---|---|---|---|
| treino | semanas 53 a 78 | 79 a 86 | treinar e testar H1 e H2 |
| teste | semanas 69 a 94 | 95 a 102 | avaliar o modelo (H3) |
| score | semanas 77 a 102 | não há | lista de risco atual |

**Regra contra vazamento:** as variáveis usam só dados da observação; o rótulo usa só as semanas seguintes. Se o modelo visse dados do futuro, o resultado ficaria falsamente bom.

**Saídas:** `data/processed/base_modelagem.csv` e `docs/dicionario_dados.md`.

#### 0. Bibliotecas e dados tratados no notebook 01

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PASTA_INTERIM = RAIZ / "data" / "interim"
PASTA_PROC = RAIZ / "data" / "processed"
PASTA_REP = RAIZ / "reports" / "variaveis"
for pasta in [PASTA_PROC, PASTA_REP]:
    pasta.mkdir(parents=True, exist_ok=True)

transacoes = pd.read_csv(PASTA_INTERIM / "transacoes.csv")
produtos = pd.read_csv(PASTA_INTERIM / "produtos.csv")
demografia = pd.read_csv(PASTA_INTERIM / "demografia.csv")
campanhas = pd.read_csv(PASTA_INTERIM / "campanhas.csv")
campanha_domicilio = pd.read_csv(PASTA_INTERIM / "campanha_domicilio.csv")
cupons_resgatados = pd.read_csv(PASTA_INTERIM / "cupons_resgatados.csv")

# só itens válidos contam como compra 
validas = transacoes[~transacoes["flag_valor_invalido"]]
SEMANA_MAX = int(transacoes["week_no"].max())
NUM = {}
print(f"{len(validas):,} linhas válidas de {len(transacoes):,}")

2,576,815 linhas válidas de 2,595,732


#### 1. Dias de cada semana
Campanhas e resgates estão em **dias**, e os cortes estão em **semanas**. Esta função converte: a semana 1 tem os dias 1 a 5 e as demais têm 7 dias.

In [2]:
def dias_da_semana(semana):
    if semana == 1:
        return 1, 5
    return 7 * semana - 8, 7 * semana - 2

# conferência contra a própria base
mapa = transacoes.groupby("week_no")["day"].agg(["min", "max"])
for s in [1, 2, 53, 78, 94]:
    print(s, dias_da_semana(s), "base:", tuple(mapa.loc[s]))

1 (1, 5) base: (1, 5)
2 (6, 12) base: (6, 12)
53 (363, 369) base: (363, 369)
78 (538, 544) base: (538, 544)
94 (650, 656) base: (650, 656)


#### 2. Função que monta a base de um corte
Recebe o nome do corte e as semanas da observação e devolve **uma linha por domicílio** que comprou na observação.

In [3]:
def construir_base(nome, obs_ini, obs_fim, janelas_perf):
    dia_ini = dias_da_semana(obs_ini)[0]
    dia_fim = dias_da_semana(obs_fim)[1]
    obs = validas[validas["week_no"].between(obs_ini, obs_fim)]
    assert obs["day"].max() <= dia_fim, "vazamento: há dia depois do fim da observação"

    # RFM e frequência de semanas
    g = obs.groupby("household_key")
    base = pd.DataFrame({
        "ultimo_dia": g["day"].max(),
        "frequencia": g["basket_id"].nunique(),     # cestas distintas
        "monetario": g["sales_value"].sum(),        # gasto na observação
        "semanas_ativas": g["week_no"].nunique(),
    })
    base["recencia_dias"] = dia_fim - base["ultimo_dia"]   # 0 = comprou no último dia
    base["ticket_medio"] = base["monetario"] / base["frequencia"]

    # diversidade: departamentos distintos comprados
    dep = obs[["household_key", "product_id"]].merge(produtos[["product_id", "department"]], on="product_id")
    dep = dep[dep["department"].fillna("").str.strip() != ""]
    base["n_departamentos"] = dep.groupby("household_key")["department"].nunique()

    # tendência de gasto: compara a 2a metade da observação com a 1a metade
    # log((gasto2 + 1) / (gasto1 + 1)): positivo = gasto crescendo; o +1 evita divisão por zero
    meio = obs_ini + (obs_fim - obs_ini + 1) // 2
    gasto1 = obs[obs["week_no"] < meio].groupby("household_key")["sales_value"].sum()
    gasto2 = obs[obs["week_no"] >= meio].groupby("household_key")["sales_value"].sum()
    base["tendencia_gasto"] = np.log((gasto2.reindex(base.index, fill_value=0) + 1) /
                                     (gasto1.reindex(base.index, fill_value=0) + 1))

    # campanhas recebidas: vigência da campanha cruza a observação
    # (a base não informa o dia em que o domicílio recebeu a campanha)
    vigentes = campanhas[(campanhas["start_day"] <= dia_fim) & (campanhas["end_day"] >= dia_ini)]["campaign"]
    expo = campanha_domicilio[campanha_domicilio["campaign"].isin(vigentes)]
    base["n_campanhas"] = expo.groupby("household_key")["campaign"].nunique()

    # cupons resgatados dentro da observação
    resg = cupons_resgatados[cupons_resgatados["day"].between(dia_ini, dia_fim)]
    base["n_resgates"] = resg.groupby("household_key").size()

    for col in ["n_departamentos", "n_campanhas", "n_resgates"]:
        base[col] = base[col].fillna(0).astype(int)
    base["recebeu_campanha"] = (base["n_campanhas"] > 0).astype(int)   # variável de H1
    base["resgatou_cupom"] = (base["n_resgates"] > 0).astype(int)      # variável de H2

    # rótulo: 1 se não houve compra válida nas p semanas seguintes
    for p in [4, 8, 12]:
        col = f"inativo_{p}s"
        if p not in janelas_perf or obs_fim + p > SEMANA_MAX:
            base[col] = np.nan          
            continue
        depois = validas[validas["week_no"].between(obs_fim + 1, obs_fim + p)]
        base[col] = (~base.index.isin(depois["household_key"])).astype(int)

    
    demo = demografia.set_index("household_key")[["income_desc", "household_size_desc"]]
    demo.columns = ["renda", "tamanho_domicilio"]
    base = base.join(demo)
    base["tem_demografia"] = base["renda"].notna().astype(int)

    base = base.drop(columns="ultimo_dia").reset_index()
    base.insert(0, "corte", nome)
    return base

#### 3. Montagem dos três cortes

In [4]:
CORTES = [("treino", 53, 78, [4, 8, 12]),
          ("teste", 69, 94, [4, 8, 12]),
          ("score", 77, 102, [])]

base = pd.concat([construir_base(*c) for c in CORTES], ignore_index=True)
base.to_csv(PASTA_PROC / "base_modelagem.csv", index=False)
print(base.shape)
base.head()

(7258, 19)


,corte,household_key,frequencia,monetario,semanas_ativas,recencia_dias,ticket_medio,n_departamentos,tendencia_gasto,n_campanhas,n_resgates,recebeu_campanha,resgatou_cupom,inativo_4s,inativo_8s,inativo_12s,renda,tamanho_domicilio,tem_demografia
0,treino,1,27,1131.00,18,4,41.888889,10,-0.178442,3,3,1,1,0.0,0.0,0.0,35-49K,2,1
1,treino,2,9,502.57,8,2,55.841111,9,1.764533,0,0,0,0,1.0,0.0,0.0,NaN,NaN,0
2,treino,3,13,504.77,12,12,38.828462,7,-0.125864,3,0,1,0,0.0,0.0,0.0,NaN,NaN,0
3,treino,4,4,77.97,4,48,19.492500,3,-0.050279,1,0,1,0,1.0,0.0,0.0,NaN,NaN,0
4,treino,5,11,164.53,8,2,14.957273,6,-0.127356,0,0,0,0,0.0,0.0,0.0,NaN,NaN,0


#### 4. Resumo por corte

In [5]:
linhas = []
for nome, g in base.groupby("corte", sort=False):
    r = {"corte": nome, "domicilios": len(g),
         "pct_recebeu_campanha": round(100 * g["recebeu_campanha"].mean(), 1),
         "pct_resgatou_cupom": round(100 * g["resgatou_cupom"].mean(), 1),
         "pct_com_demografia": round(100 * g["tem_demografia"].mean(), 1),
         "monetario_total": round(float(g["monetario"].sum()), 2)}
    for p in [4, 8, 12]:
        s = g[f"inativo_{p}s"].dropna()
        r[f"inativos_{p}s"] = int(s.sum()) if len(s) else None
        r[f"pct_inativos_{p}s"] = round(100 * s.mean(), 1) if len(s) else None
    linhas.append(r)
resumo = pd.DataFrame(linhas).set_index("corte")
NUM["resumo_cortes"] = {k: {kk: (None if pd.isna(vv) else float(vv)) for kk, vv in v.items()}
                        for k, v in resumo.to_dict("index").items()}
resumo

,domicilios,pct_recebeu_campanha,pct_resgatou_cupom,pct_com_demografia,monetario_total,inativos_4s,pct_inativos_4s,inativos_8s,pct_inativos_8s,inativos_12s,pct_inativos_12s
corte,,,,,,,,,,,
treino,2409,57.6,12.3,33.3,2288669.18,420.0,17.4,213.0,8.8,141.0,5.9
teste,2417,57.7,14.0,33.1,2311939.71,429.0,17.7,244.0,10.1,NaN,NaN
score,2432,57.2,11.9,32.9,2328852.42,NaN,NaN,NaN,NaN,NaN,NaN


#### 5. Conciliação com o notebook 01
O notebook 01 mediu a viabilidade com todas as linhas e aqui só contam compras válidas. A diferença deve ser mínima.

In [6]:
viab = json.load(open(RAIZ / "reports" / "perfil" / "numeros_perfil.json", encoding="utf-8"))["viabilidade_janelas"]
conc = []
for v in viab:
    nome = {"53-78": "treino", "69-94": "teste"}[v["obs"]]
    g = base[base["corte"] == nome]
    conc.append({"corte": nome, "perf": v["perf_semanas"], "ativos_nb01": v["ativos"], "ativos_nb02": len(g),
                 "inativos_nb01": v["inativos"], "inativos_nb02": int(g[f"inativo_{v['perf_semanas']}s"].sum())})
NUM["conciliacao_nb01"] = conc
pd.DataFrame(conc)

,corte,perf,ativos_nb01,ativos_nb02,inativos_nb01,inativos_nb02
0,treino,4,2409,2409,419,420
1,treino,8,2409,2409,211,213
2,treino,12,2409,2409,141,141
3,teste,4,2417,2417,429,429
4,teste,8,2417,2417,244,244


#### 6. Verificação de 3 domicílios, calculados à mão
Recalculando as variáveis de um inativo, de um ativo e de um que resgatou cupom direto das transações, sem usar a função. Se algum valor não bater, o `assert` interrompe o notebook.

In [7]:
tr = base[base["corte"] == "treino"]
rng = np.random.default_rng(42)   # semente fixa
escolhidos = {
    "inativo": int(rng.choice(tr.loc[tr["inativo_8s"] == 1, "household_key"])),
    "ativo": int(rng.choice(tr.loc[tr["inativo_8s"] == 0, "household_key"])),
    "resgatou": int(rng.choice(tr.loc[tr["resgatou_cupom"] == 1, "household_key"])),
}
dia_ini, dia_fim = dias_da_semana(53)[0], dias_da_semana(78)[1]
dia_fim_perf = dias_da_semana(78 + 8)[1]
vigentes = campanhas[(campanhas["start_day"] <= dia_fim) & (campanhas["end_day"] >= dia_ini)]["campaign"]

manual = []
for tipo, hh in escolhidos.items():
    compras = validas[validas["household_key"] == hh]
    na_obs = compras[compras["day"].between(dia_ini, dia_fim)]
    depois = compras[(compras["day"] > dia_fim) & (compras["day"] <= dia_fim_perf)]
    camp_hh = campanha_domicilio[(campanha_domicilio["household_key"] == hh) & campanha_domicilio["campaign"].isin(vigentes)]
    resg_hh = cupons_resgatados[(cupons_resgatados["household_key"] == hh) & cupons_resgatados["day"].between(dia_ini, dia_fim)]
    m = {"tipo": tipo, "household_key": hh,
         "frequencia": int(na_obs["basket_id"].nunique()),
         "monetario": round(float(na_obs["sales_value"].sum()), 2),
         "recencia_dias": int(dia_fim - na_obs["day"].max()),
         "n_campanhas": int(camp_hh["campaign"].nunique()),
         "n_resgates": len(resg_hh),
         "inativo_8s": int(depois.empty)}
    calc = tr[tr["household_key"] == hh].iloc[0]
    for k in ["frequencia", "recencia_dias", "n_campanhas", "n_resgates", "inativo_8s"]:
        assert m[k] == calc[k], (tipo, k, m[k], calc[k])
    assert abs(m["monetario"] - calc["monetario"]) < 0.01
    manual.append(m)
NUM["verificacao_manual"] = manual
pd.DataFrame(manual)

,tipo,household_key,frequencia,monetario,recencia_dias,n_campanhas,n_resgates,inativo_8s
0,inativo,187,10,108.65,1,1,0,1
1,ativo,1937,52,1922.82,1,2,12,0
2,resgatou,1753,142,1058.81,0,3,2,0


#### 7. Estatísticas descritivas e correlação (treino)

In [8]:
num_cols = ["recencia_dias", "frequencia", "monetario", "ticket_medio", "n_departamentos",
            "tendencia_gasto", "n_campanhas", "n_resgates"]
desc = tr[num_cols].describe(percentiles=[.25, .5, .75, .99]).T.round(2)
desc.to_csv(PASTA_REP / "descritivas_treino.csv")
desc

,count,mean,std,min,25%,50%,75%,99%,max
recencia_dias,2409.0,16.45,28.14,0.00,1.00,6.00,17.00,139.92,181.00
frequencia,2409.0,32.54,37.06,1.00,10.00,22.00,42.00,174.00,469.00
monetario,2409.0,950.05,1049.83,1.12,225.83,594.58,1320.26,4848.98,13234.72
ticket_medio,2409.0,31.56,21.95,1.00,16.68,26.31,40.63,111.08,214.09
n_departamentos,2409.0,8.97,3.52,1.00,7.00,9.00,11.00,17.00,19.00
tendencia_gasto,2409.0,0.02,1.47,-7.03,-0.47,-0.00,0.43,5.18,6.64
n_campanhas,2409.0,1.66,1.86,0.00,0.00,1.00,3.00,6.92,10.00
n_resgates,2409.0,0.46,1.67,0.00,0.00,0.00,0.00,9.00,23.00


In [9]:
# Correlação de Spearman 
# Acima de 0,8 em módulo: duas variáveis quase repetem a mesma informação.
corr = tr[num_cols].corr(method="spearman").round(2)
altas = []
for i, a in enumerate(num_cols):
    for b in num_cols[i + 1:]:
        if abs(corr.loc[a, b]) > 0.8:
            altas.append({"a": a, "b": b, "rho": float(corr.loc[a, b])})
NUM["correlacoes_altas_spearman"] = altas
corr

,recencia_dias,frequencia,monetario,ticket_medio,n_departamentos,tendencia_gasto,n_campanhas,n_resgates
recencia_dias,1.00,-0.63,-0.58,-0.11,-0.51,-0.23,-0.43,-0.24
frequencia,-0.63,1.00,0.85,0.05,0.75,-0.00,0.67,0.33
monetario,-0.58,0.85,1.00,0.52,0.84,0.00,0.69,0.39
ticket_medio,-0.11,0.05,0.52,1.00,0.41,0.03,0.22,0.20
n_departamentos,-0.51,0.75,0.84,0.41,1.00,0.01,0.58,0.34
tendencia_gasto,-0.23,-0.00,0.00,0.03,0.01,1.00,-0.12,-0.00
n_campanhas,-0.43,0.67,0.69,0.22,0.58,-0.12,1.00,0.40
n_resgates,-0.24,0.33,0.39,0.20,0.34,-0.00,0.40,1.00


#### 8. Dicionário de dados

In [10]:
DICIONARIO = {
    "corte": "Nome do corte temporal (treino, teste ou score).",
    "household_key": "Identificador do domicílio.",
    "frequencia": "Cestas distintas com pelo menos um item válido na observação.",
    "monetario": "Soma de sales_value dos itens válidos na observação.",
    "semanas_ativas": "Semanas distintas com compra na observação.",
    "recencia_dias": "Dias entre a última compra e o último dia da observação (0 = comprou no último dia).",
    "ticket_medio": "monetario / frequencia.",
    "n_departamentos": "Departamentos distintos comprados (diversidade de compra).",
    "tendencia_gasto": "log((gasto 2a metade + 1) / (gasto 1a metade + 1)); positivo = gasto crescendo.",
    "n_campanhas": "Campanhas recebidas cuja vigência cruza a observação.",
    "n_resgates": "Cupons resgatados com dia dentro da observação.",
    "recebeu_campanha": "1 se n_campanhas > 0 (variável de H1).",
    "resgatou_cupom": "1 se n_resgates > 0 (variável de H2).",
    "inativo_4s": "1 se não houve compra válida nas 4 semanas após a observação.",
    "inativo_8s": "1 se não houve compra válida nas 8 semanas após a observação (rótulo principal).",
    "inativo_12s": "1 se não houve compra válida nas 12 semanas após a observação; vazio se a janela passa do fim da base.",
    "renda": "Faixa de renda (INCOME_DESC); vazio sem demografia.",
    "tamanho_domicilio": "Tamanho do domicílio (HOUSEHOLD_SIZE_DESC); vazio sem demografia.",
    "tem_demografia": "1 se o domicílio está em hh_demographic.",
}
linhas = [f"| `{c}` | {base[c].dtype} | {DICIONARIO.get(c, '')} | {int(base[c].isna().sum())} |" for c in base.columns]
texto = ("# Dicionário de dados: base_modelagem\n\n"
         "Grão: uma linha por domicílio e corte. Gerado por `notebooks/02_rotulagem_e_variaveis.ipynb`.\n\n"
         "| Coluna | Tipo | Definição | Ausentes |\n|---|---|---|---|\n" + "\n".join(linhas) + "\n")
(RAIZ / "docs").mkdir(exist_ok=True)
(RAIZ / "docs" / "dicionario_dados.md").write_text(texto, encoding="utf-8")
print(texto)

# Dicionário de dados: base_modelagem

Grão: uma linha por domicílio e corte. Gerado por `notebooks/02_rotulagem_e_variaveis.ipynb`.

| Coluna | Tipo | Definição | Ausentes |
|---|---|---|---|
| `corte` | str | Nome do corte temporal (treino, teste ou score). | 0 |
| `household_key` | int64 | Identificador do domicílio. | 0 |
| `frequencia` | int64 | Cestas distintas com pelo menos um item válido na observação. | 0 |
| `monetario` | float64 | Soma de sales_value dos itens válidos na observação. | 0 |
| `semanas_ativas` | int64 | Semanas distintas com compra na observação. | 0 |
| `recencia_dias` | int64 | Dias entre a última compra e o último dia da observação (0 = comprou no último dia). | 0 |
| `ticket_medio` | float64 | monetario / frequencia. | 0 |
| `n_departamentos` | int64 | Departamentos distintos comprados (diversidade de compra). | 0 |
| `tendencia_gasto` | float64 | log((gasto 2a metade + 1) / (gasto 1a metade + 1)); positivo = gasto crescendo. | 0 |
| `n_campanhas` | int64 

#### 9. Gravação dos números e achados

In [11]:
with open(PASTA_REP / "numeros_variaveis.json", "w", encoding="utf-8") as f:
    json.dump(NUM, f, ensure_ascii=False, indent=2)

for nome, r in NUM["resumo_cortes"].items():
    txt = f"{nome}: {int(r['domicilios']):,} domicílios ativos; campanha {r['pct_recebeu_campanha']}%, resgate {r['pct_resgatou_cupom']}%"
    if r["pct_inativos_8s"] is not None:
        txt += f"; inativos em 8 semanas: {int(r['inativos_8s'])} ({r['pct_inativos_8s']}%)"
    print(txt + ".")
print("Correlações altas entre variáveis:", NUM["correlacoes_altas_spearman"])

treino: 2,409 domicílios ativos; campanha 57.6%, resgate 12.3%; inativos em 8 semanas: 213 (8.8%).
teste: 2,417 domicílios ativos; campanha 57.7%, resgate 14.0%; inativos em 8 semanas: 244 (10.1%).
score: 2,432 domicílios ativos; campanha 57.2%, resgate 11.9%.
Correlações altas entre variáveis: [{'a': 'frequencia', 'b': 'monetario', 'rho': 0.85}, {'a': 'monetario', 'b': 'n_departamentos', 'rho': 0.84}]
